# DEFT-DPT + EgoHAnG — 15-minute walkthrough

Same clip, two questions:
**what is happening** (DEFT-DPT, recognition) and **what happens next** (EgoHAnG, anticipation).

Run every cell once before the talk and commit this notebook *with outputs*,
so you have a fallback if the live run stalls.


## 0 — Setup  *(~1:30)*
What is on disk, and where it came from.

In [ ]:
import subprocess, sys, torch
print("python  ", sys.version.split()[0])
print("torch   ", torch.__version__, "| cuda:", torch.cuda.is_available())
!ls -la SavedModels/ Features/ | head -20

These features were produced once, offline, with:

```bash
python deft_dpt.py extract --rgb_root $DATA_ROOT/EPIC_Kitchen/RGB/P01_04 \
    --flow_u_root .../u --flow_v_root .../v \
    --labels .../P01_04.csv --out Features/Feature_P01_04_EpicKitchen.csv
```

We load them here rather than re-extracting on stage.

## 1 — What DEFT does  *(~2:30)*
One clip, four panels: input, affine warp, weight map, output.

In [ ]:
!python deft_dpt.py deft --frames Input_Data/RGB/P01_04 --n 3 --out Results/deft_demo.png

In [ ]:
from IPython.display import Image
Image("Results/deft_demo.png", width=1000)

Read the printed `theta`. If it is the identity for every frame, the affine
branch is inactive and the radial weighting is doing the work — say so plainly
rather than letting it come up in Q&A.

## 2 — What DPT does  *(~1:30)*
Sparsity adapts per clip instead of a fixed K.

In [ ]:
!python deft_dpt.py graph --features Features/Feature_P01_04_EpicKitchen.csv \
    --meta_cols 4 --max_nodes 60 --out Results/dpt_graph.png

In [ ]:
Image("Results/dpt_graph.png", width=1100)

## 3 — Recognition: one sample  *(~2:00)*
**What is happening in this clip?**

In [ ]:
!python deft_dpt.py demo --features Features/Feature_P01_04_EpicKitchen.csv \
    --model_path SavedModels/best_model.pth --meta_cols 4 --index 0

## 4 — Anticipation: the same clip  *(~2:30)*
**What happens next?** — EgoHAnG, horizons 2.0s down to 0.25s.

In [ ]:
!cd ../EgoHANG && python evaluate.py --dataset epic_kitchens \
    --fused_csv EPIC-Kitchens/Features/P01_04_fused_features_PCA.csv \
    --label_csv EPIC-Kitchens/Labels/P01_04.csv \
    --model_path checkpoints/P01_04_Fused_model.pth \
    --subset 32

## 5 — Full evaluation  *(~2:30)*
Live subset run, then the full-test-set numbers from the paper.

In [ ]:
!python deft_dpt.py evaluate --features Features/Feature_P01_04_EpicKitchen.csv \
    --model_path SavedModels/best_model.pth --meta_cols 4 \
    --subset 200 --seed 42 --save_json Results/subset_metrics.json

In [ ]:
import json, pandas as pd
# Full-test-set results, computed offline. Keep this file in the repo.
full = json.load(open("Results/full_eval.json"))
pd.DataFrame([full])

**Say this out loud:** the live run is a 200-frame subset for time; the table
is the full test set. Also state the split protocol — random over frames, not
segment-disjoint.

## 6 — Qualitative results and failure cases  *(~4:00)*

In [ ]:
Image("Results/Qualitative_TIM.png", width=1000)

In [ ]:
Image("SupplementaryMaterials/FailureCase.png", width=1000)

## 7 — Reproduce  *(~2:00 buffer)*

```bash
git clone https://github.com/pawanesh-mnnit/deft_dpt && cd deft_dpt
pip install -r requirements.txt
bash scripts/download_assets.sh
python deft_dpt.py evaluate --features Features/Feature_P01_04_EpicKitchen.csv \
    --model_path SavedModels/best_model.pth --meta_cols 4
```

Put a QR code to each repo on the final slide — people photograph it.